# CertVIC main-200 — Open-VLM eval (Kaggle T4×2) — AFTER GATES ONLY

> **Do not run until ALL pass:** quality gates · detectability **AUC < 0.80** ·
> human review (`pilot_eval_tasks_reviewed.jsonl`) · item certificates.
> `run_eval --evidence-run` refuses unreviewed tasks and non-open-local providers.

Two workers in parallel — **GPU 0 → shard 0**, **GPU 1 → shard 1** — per provider.

**Estimated runtime (T4×2):** ~10–15 min for 1 provider — 4-bit load ~2–3 min, then
~103 task-pairs (206 inferences) split over the 2 GPUs at ~3–4 s each. Each extra
provider adds ~10–15 min.

## Attach inputs
- **certvic** bundle (this `.zip`) — auto-detected wherever it mounts.
- **edits dataset** — upload `data/edits/main_real_200/` (it holds `<id>.jpg` + `orig/<id>.jpg`)
  and drop `pilot_eval_tasks_reviewed.jsonl` beside it. Both original and edited images
  live here now (re-encoded identically by the local ingest — no separate ADE20K needed).
- **VLM weights** — a Kaggle Model ref or open HF id (the launch cell fetches it; set
  `HF_TOKEN` in Add-ons > Secrets only if you pick a gated model).

Settings: **GPU T4 ×2**, **Internet = On** for run 1 (weights download).

In [ ]:
# --- deps + bundle/reviewed-tasks discovery + path remap (all slug-agnostic) ---
import os, sys, json, glob, importlib.util
from pathlib import Path

# VLM deps. bitsandbytes is REQUIRED (4-bit) to fit a 7B VLM on a 14.5 GB T4 -> install if absent.
import subprocess
_missing = [m for m in ("accelerate", "bitsandbytes") if importlib.util.find_spec(m) is None]
if _missing:
    print("installing", _missing, "(needs Internet On)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *_missing], check=False)
    importlib.invalidate_caches()
for m in ("torch", "transformers", "accelerate", "bitsandbytes"):
    print(m, "OK" if importlib.util.find_spec(m) else "MISSING -> %pip install -U " + m)

def find_certvic():
    h = glob.glob("/kaggle/input/**/main_real_200/gpu_shards/pilot_edit_plan_shard0_of_2.jsonl", recursive=True)
    if h:
        return h[0].split("/data/results/")[0]
    z = glob.glob("/kaggle/input/**/certvic_kaggle_main200_bundle.zip", recursive=True)
    if z:
        import zipfile
        with zipfile.ZipFile(z[0]) as zf:
            zf.extractall("/kaggle/working/certvic_bundle")
        return "/kaggle/working/certvic_bundle"
    r = glob.glob("/kaggle/input/**/README_KAGGLE_BUNDLE.md", recursive=True)
    if r:
        return os.path.dirname(r[0])
    raise FileNotFoundError("Attach the certvic bundle dataset.")
CERTVIC = find_certvic(); sys.path.insert(0, CERTVIC)
WORK = Path("/kaggle/working")

_t = glob.glob("/kaggle/input/**/pilot_eval_tasks_reviewed.jsonl", recursive=True)
assert _t, "Add pilot_eval_tasks_reviewed.jsonl (produced AFTER human review)."
rows = [json.loads(l) for l in open(_t[0])]

# Both original (orig/<id>.jpg) and edited (<id>.jpg) live in the edits dataset. Detect the
# local edits root from the tasks and the Kaggle mount (dir that contains the 'orig' subdir):
LOCAL_EDITS = os.path.commonpath([rows[0]["original_image_path"], rows[0]["edited_image_path"]])
_o = glob.glob("/kaggle/input/**/orig", recursive=True)
EDITS_KAGGLE = os.environ.get("EDITS_ROOT") or (os.path.dirname(_o[0]) if _o else None)
assert EDITS_KAGGLE, "Attach the edits dataset (data/edits/main_real_200 with <id>.jpg + orig/<id>.jpg)."
for r in rows:
    for k in ("original_image_path", "edited_image_path"):
        if r.get(k):
            r[k] = r[k].replace(LOCAL_EDITS, EDITS_KAGGLE)
# run_eval loads the strict TaskItem schema (nested source+edit, extra=forbid). The
# materialized/reviewed rows are the richer preview format -> project them onto TaskItem.
if rows and "source" not in rows[0]:
    from certvic.schema import TaskItem
    from certvic.schema.edit import EditSpec
    from certvic.schema.source import SourceImageRecord
    def _to_taskitem(r):
        src = SourceImageRecord(source_id=r["source_id"], source_name="ADE20K", license_category="pointer_only")
        ed = EditSpec(edit_id=r["edit_id"], source_id=r["source_id"], edit_type=r["edit_type"],
                      task_family=r["task_family"], domain=r["domain"], expected_effect=r["expected_effect"])
        m = dict(r.get("metadata") or {}); m.setdefault("evidence_status", r.get("evidence_status", "HUMAN_REVIEWED_NON_EVIDENCE"))
        return TaskItem(item_id=r["item_id"], source=src, edit=ed,
                        original_image_path=r["original_image_path"], edited_image_path=r["edited_image_path"],
                        question_original=r["question_original"], question_edited=r["question_edited"],
                        answer_original=r["answer_original"], answer_edited=r["answer_edited"],
                        required_change=r["required_change"], answer_format=r["answer_format"],
                        task_family=r["task_family"], domain=r["domain"], split=r["split"], metadata=m)
    rows = [json.loads(_to_taskitem(r).model_dump_json()) for r in rows]
open(WORK / "tasks_reviewed.jsonl", "w").writelines(json.dumps(r) + "\n" for r in rows)
miss = sum(1 for r in rows for k in ("original_image_path", "edited_image_path") if not os.path.exists(r[k]))
ev = sorted({r.get("metadata", {}).get("evidence_status") or r.get("evidence_status") for r in rows})
print(len(rows), "reviewed tasks | missing images:", miss, "(must be 0) | evidence:", ev)

In [ ]:
%%writefile /kaggle/working/provider_patch.py
# Real VLM answer(), patched onto certvic OpenVLMProvider. run_eval keeps all
# leakage/evidence/resume logic; only model loading + generation are new.
import os, importlib.util, torch
from PIL import Image
import certvic.providers.open_vlm as ovlm
WEIGHTS = os.environ["WEIGHTS"]; PROVIDER = os.environ["PROVIDER"]

def _qwen():
    from transformers import AutoProcessor, BitsAndBytesConfig
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration as Model
    except Exception:
        from transformers import AutoModelForImageTextToText as Model
    # 4-bit is REQUIRED on a 14.5 GB T4 (fp16 7B ~15 GB OOMs). Must go through
    # BitsAndBytesConfig -- the bare load_in_4bit kwarg is ignored in recent transformers.
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = Model.from_pretrained(WEIGHTS, device_map={"": 0},
                                  quantization_config=bnb, low_cpu_mem_usage=True)
    proc = AutoProcessor.from_pretrained(WEIGHTS, max_pixels=768 * 768)  # cap vision tokens -> less VRAM
    @torch.inference_mode()
    def answer(self, image_path, prompt):
        img = Image.open(image_path).convert("RGB")
        msgs = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=[img], return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
        return proc.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return answer

LOADERS = {"qwen2_5_vl_7b": _qwen}                        # add internvl_8b / llava_onevision_7b similarly
ovlm.OpenVLMProvider.load = lambda self: None
ovlm.OpenVLMProvider.answer = LOADERS[PROVIDER]()
print("patched OpenVLMProvider.answer for", PROVIDER)

In [ ]:
%%writefile /kaggle/working/worker_vlm.py
import os, sys
sys.path.insert(0, os.environ["CERTVIC"])
exec(open("/kaggle/working/provider_patch.py").read())
from certvic.eval.run_eval import main as run_eval_main
shard = int(os.environ["SHARD"]); provider = os.environ["PROVIDER"]
run_eval_main([
    "--config", f"{os.environ['CERTVIC']}/configs/kaggle_open_vlm.yaml",
    "--tasks", "/kaggle/working/tasks_reviewed.jsonl",
    "--out", f"/kaggle/working/pred_{provider}_shard{shard}.jsonl",
    "--provider", provider, "--run-id", f"main200_{provider}_shard{shard}",
    "--shard-index", str(shard), "--num-shards", "2",
    "--strict-leakage", "--evidence-run", "--fail-fast"])

In [ ]:
# --- per provider: GPU0 shard0 + GPU1 shard1 in parallel, with LIVE progress ---
import os, sys, time, glob, subprocess
PROVIDERS = {
    "qwen2_5_vl_7b": "Qwen/Qwen2.5-VL-7B-Instruct",    # mounted dir OR open HF id
    # "internvl_8b": "OpenGVLab/InternVL2-8B",
    # "llava_onevision_7b": "llava-hf/llava-onevision-qwen2-7b-ov-hf",
}
_HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
def ensure_local(repo):
    if os.path.exists(repo):
        return repo
    hits = glob.glob(f"/kaggle/input/**/{repo.split('/')[-1]}*/config.json", recursive=True)
    if hits:
        return os.path.dirname(hits[0])                  # mounted snapshot (no HF call)
    from huggingface_hub import snapshot_download        # ~16 GB, Internet ON; HF_TOKEN if gated
    return snapshot_download(repo, local_dir=f"/kaggle/working/{repo.split('/')[-1]}", token=_HF_TOKEN)
PROVIDERS = {name: ensure_local(src) for name, src in PROVIDERS.items()}
print("resolved provider weights:", PROVIDERS, flush=True)
EXPECT = 2 * sum(1 for _ in open("/kaggle/working/tasks_reviewed.jsonl"))   # 2 variants/task

def run_shard(provider, weights, gpu, shard):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "SHARD": str(shard),
           "PROVIDER": provider, "WEIGHTS": weights, "CERTVIC": CERTVIC, "PYTHONUNBUFFERED": "1",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    return subprocess.Popen([sys.executable, "-u", "/kaggle/working/worker_vlm.py"], env=env,
                            stdout=open(f"/kaggle/working/log_{provider}_s{shard}.txt", "w"),
                            stderr=subprocess.STDOUT)

for provider, weights in PROVIDERS.items():
    procs = {0: run_shard(provider, weights, 0, 0), 1: run_shard(provider, weights, 1, 1)}
    t0 = time.time()
    print(f"[{provider}] launched; 4-bit load ~2-3 min before first predictions", flush=True)
    while any(p.poll() is None for p in procs.values()):
        time.sleep(20)
        line = []
        for s in (0, 1):
            f = f"/kaggle/working/pred_{provider}_shard{s}.jsonl"
            n = sum(1 for _ in open(f)) if os.path.exists(f) else 0
            line.append(f"GPU{s} {n} preds {'run' if procs[s].poll() is None else 'done'}")
        print(f"[{int(time.time()-t0):4d}s {provider}] ~{EXPECT} total | " + " | ".join(line), flush=True)
    # Auto-recover: if a shard died (transient CUBLAS/OOM from launch-time contention),
    # retry it ALONE -- the other GPU is free now and run_eval resumes from what it wrote.
    for s in (0, 1):
        if procs[s].returncode not in (0, None):
            print(f"[{provider}] shard{s} exited {procs[s].returncode}; retrying alone (resume-safe)...", flush=True)
            rp = run_shard(provider, weights, s, s)
            while rp.poll() is None:
                time.sleep(20)
                f = f"/kaggle/working/pred_{provider}_shard{s}.jsonl"
                print(f"  retry shard{s}: {sum(1 for _ in open(f)) if os.path.exists(f) else 0} preds", flush=True)
            print(f"[{provider}] shard{s} retry exit {rp.returncode}", flush=True)
    for s in (0, 1):
        print(f"--- {provider} shard{s} log tail ---\n" + open(f"/kaggle/working/log_{provider}_s{s}.txt").read()[-500:])

In [ ]:
# --- merge predictions + package preds + logs + run-manifests into ONE zip ---
import os, zipfile
total = 0
for provider in PROVIDERS:
    merged = []
    for s in (0, 1):
        f = f"/kaggle/working/pred_{provider}_shard{s}.jsonl"
        if os.path.exists(f):
            lines = list(open(f)); merged += lines; total += len(lines)
        else:
            print("WARNING: no predictions from", f, "- check that shard's log below (likely OOM/load error).")
    open(f"/kaggle/working/pred_{provider}_merged.jsonl", "w").writelines(merged)
with zipfile.ZipFile("/kaggle/working/vlm_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("/kaggle/working"):
        if (f.startswith("pred_") and f.endswith(".jsonl")) or (f.startswith("log_") and f.endswith(".txt")) \
           or f.endswith(".run_manifest.json"):
            z.write(f"/kaggle/working/{f}", f)
print(f"predictions written: {total} | DOWNLOAD: vlm_out.zip (preds + logs + run manifests)")

## Back on the Mac — score & certify
Concatenate `pred_*_merged.jsonl` → `merged.jsonl`, then run
`certvic.eval.output_triage`, `certvic.metrics.score_predictions`,
`certvic.reporting.build_v2_report`. `score_summary.json` carries `a`, `p`,
`Δ = a − p` and the anytime-valid CS. Certified claims only after all gates pass.